In [3]:
import httpx
from dotenv import load_dotenv
load_dotenv()
from anthropic import Anthropic
client = Anthropic()
model = "claude-sonnet-4-0"

http_client = httpx.Client(verify=False)
client = Anthropic(http_client=http_client)

from tavily import TavilyClient

tavily = TavilyClient("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")

# %%
# Define the Search Tool function
def run_tavily_search(query):
    print(f"--- Executing Tavily Search for: {query} ---")
    # We limit to 3 results for brevity
    response = tavily.search(query=query, max_results=3)
    return str(response['results'])

# %%
# Define the Tool Schema for Claude
tools = [
    {
        "name": "web_search",
        "description": "A tool to search the internet for real-time information, news, and facts.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query to look up on the internet"
                }
            },
            "required": ["query"]
        }
    }
]


# %%
# Initial User Prompt
messages = [
    {"role": "user", "content": "Who won the Super Bowl in 2024 and what was the score?"}
]

# %%
# First Call: Ask Claude (Claude will likely request the tool)
response = client.messages.create(
    model=model,
    max_tokens=1024,
    tools=tools,
    messages=messages
)

response

# %%
# Process Tool Call and Get Results
if response.stop_reason == "tool_use":
    # Add Claude's tool request to the conversation history
    messages.append({"role": "assistant", "content": response.content})
    
    # Find the tool use block
    tool_use = next(block for block in response.content if block.type == "tool_use")
    tool_name = tool_use.name
    tool_input = tool_use.input
    tool_use_id = tool_use.id

    if tool_name == "web_search":
        # Execute the actual Tavily search
        search_result = run_tavily_search(tool_input["query"])

        # Add the search result to the conversation history
        messages.append({
            "role": "user",
            "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": tool_use_id,
                    "content": search_result
                }
            ]
        })


# %%
# Final Call: Send results back to Claude for the final answer
final_response = client.messages.create(
            model=model,
            max_tokens=1024,
            tools=tools,
            messages=messages
        )
        
print("\nClaude's Final Answer:")
print(final_response.content[0].text)

C:\Users\61093123\AppData\Local\Temp\ipykernel_26408\1700731328.py:51: DeprecationWarning: The model 'claude-sonnet-4-0' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client.messages.create(


--- Executing Tavily Search for: Super Bowl 2024 winner score ---


C:\Users\61093123\AppData\Local\Temp\ipykernel_26408\1700731328.py:91: DeprecationWarning: The model 'claude-sonnet-4-0' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  final_response = client.messages.create(



Claude's Final Answer:
The **Kansas City Chiefs** won Super Bowl LVIII (58) in 2024, defeating the **San Francisco 49ers** with a final score of **25-22 in overtime**.

This historic game took place on February 11, 2024, at Allegiant Stadium in Las Vegas. The Chiefs' victory made them back-to-back Super Bowl champions and marked their third Super Bowl title in five years. The game-winning touchdown was scored by Mecole Hardman on a 3-yard pass from Patrick Mahomes during overtime, making this the second Super Bowl ever to go to overtime.
